# Grafo principal y subgrafo de coautoría

Los dos grafos se construyen con `Canonico_FINAL.csv` de 2024–2025. 

| Grafo | Nodos | Aristas |
|---|---|---|
| Principal | Artículo, autor, afiliación y keyword | Autor–artículo, autor–afiliación y artículo–keyword |
| Coautoría | Autores, con sus afiliaciones como atributos | Dos autores que aparecen en un mismo artículo |

## 1. Leer la base final

Los dos archivos GML y las dos imágenes se guardan en `06_grafos/`.

In [ ]:
import os
import json
import hashlib
from itertools import combinations

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

fuente = "04_Limpieza/06_Finales/Canonico_FINAL.csv"
archivo = "../" + fuente
carpeta_salida = "../06_grafos"

# Si se ejecuta desde la raíz del proyecto en lugar de notebooks/
if not os.path.exists(archivo):
    archivo = fuente
    carpeta_salida = "06_grafos"

datos = pd.read_csv(archivo, dtype=str, keep_default_na=False, encoding="utf-8-sig")

with open(archivo, "rb") as f:
    huella = hashlib.sha256(f.read()).hexdigest()

print(f"Entrada: {fuente} ({len(datos):,} filas)")
print("SHA-256:", huella)

## 2. Comprobar artículos y participaciones

La clave de una participación es `indice` junto con `Autor_norm`. Antes de conservar una fila por artículo, se revisa que los metadatos coincidan en todas sus filas.

In [ ]:
campos_articulo = ["Titulo", "Año", "ISBN", "ISSN", "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"]

# Revisión básica antes de armar los grafos
print("Filas sin título:", (datos["Titulo"].str.strip() == "").sum())
print("Filas sin autor:", (datos["Autor_norm"].str.strip() == "").sum())
print("Pares indice-autor repetidos:", datos.duplicated(["indice", "Autor_norm"]).sum())

diferencias = datos.groupby("indice")[campos_articulo].nunique() > 1
print("Artículos con metadatos distintos entre filas:", diferencias.any(axis=1).sum())

articulos = datos.drop_duplicates("indice")
print(f"\nArtículos: {len(articulos):,} | Nombres de autor distintos: {datos['Autor_norm'].nunique():,}")
print("\nArtículos por año:")
print(articulos["Año"].value_counts().sort_index())

# Revisar si hay títulos repetidos con distinto índice
mismo_titulo = articulos["Titulo"].duplicated(keep=False)
if mismo_titulo.any():
    display(articulos.loc[mismo_titulo, ["indice", "Titulo", "Doi"]].sort_values("Titulo"))

## 3. Construir el grafo principal

Cada artículo se identifica por `indice` y las palabras clave se separan por punto y coma conservando su escritura.

In [ ]:
G_principal = nx.Graph(
    nombre="Grafo bibliográfico principal",
    fuente=fuente,
    sha256=huella,
    criterioArticulo="Un nodo por indice del CSV",
    criterioAutor="Coincidencia exacta de Autor_norm",
    criterioKeyword="Separación por punto y coma; se conserva la escritura"
)

for _, fila in articulos.iterrows():
    nodo_art = "articulo:" + fila["indice"]
    G_principal.add_node(
        nodo_art,
        tipo="articulo",
        nombre=fila["Titulo"],
        indice=int(fila["indice"]),
        titulo=fila["Titulo"],
        anio=int(fila["Año"]),
        isbn=fila["ISBN"],
        issn=fila["ISSN"],
        doi=fila["Doi"],
        url=fila["URL"],
        area=fila["Area"],
        subarea=fila["SubArea"],
        keywords=fila["Keywords"],
        abstract=fila["Abstract"]
    )

    for termino in fila["Keywords"].split(";"):
        palabra = termino.strip()
        if palabra != "":
            nodo_kw = "keyword:" + palabra
            G_principal.add_node(nodo_kw, tipo="keyword", nombre=palabra)
            G_principal.add_edge(nodo_art, nodo_kw, tipo="keyword")

Cada arista autor–artículo guarda las dos afiliaciones de esa participación, y cada arista autor–afiliación reúne los índices de los artículos que respaldan esa afiliación en formato JSON.

In [ ]:
def lista_json(valores):
    if len(valores) == 0:
        return "[ ]"  # Para que GML guarde la lista vacía como texto
    return json.dumps(sorted(valores), ensure_ascii=False)

for nombre, filas in datos.groupby("Autor_norm"):
    nodo_autor = "autor:" + nombre
    respaldo = {}  # {afiliacion: conjunto de indices}

    for _, fila in filas.iterrows():
        nodo_art = "articulo:" + fila["indice"]
        G_principal.add_edge(
            nodo_autor,
            nodo_art,
            tipo="autoria",
            afiliacion1=fila["Afiliacion1"],
            afiliacion2=fila["Afiliacion2"]
        )

        for dep in [fila["Afiliacion1"], fila["Afiliacion2"]]:
            if dep.strip() != "":
                if dep not in respaldo:
                    respaldo[dep] = set()
                respaldo[dep].add(int(fila["indice"]))

    indices_autor = [int(i) for i in filas["indice"]]
    G_principal.add_node(
        nodo_autor,
        tipo="autor",
        nombre=nombre,
        afiliaciones=lista_json(respaldo.keys()),
        articulos=lista_json(indices_autor),
        numeroArticulos=len(filas)
    )

    for dep, indices_dep in respaldo.items():
        nodo_afil = "afiliacion:" + dep
        G_principal.add_node(nodo_afil, tipo="afiliacion", nombre=dep)
        G_principal.add_edge(nodo_autor, nodo_afil, tipo="afiliacion", articulos=lista_json(indices_dep))

## 4. Construir el subgrafo de coautoría

Se incluyen todos los autores (también los aislados). Cada arista guarda en `articulos` los índices compartidos y en `weight` cuántos son.

In [ ]:
G_coautoria = nx.Graph(
    nombre="Subgrafo de coautoría",
    fuente=fuente,
    sha256=huella,
    criterio="Proyección sobre autores por indice de artículo compartido",
    significadoPeso="Número de índices de artículo compartidos"
)

# Copiamos solo los nodos de tipo autor
for nodo, attrs in G_principal.nodes(data=True):
    if attrs["tipo"] == "autor":
        G_coautoria.add_node(nodo, **attrs)

# Juntamos las parejas de autores que comparten cada artículo
compartidos = {}
for indice in articulos["indice"]:
    nodo_art = "articulo:" + indice
    autores_art = []
    for vecino in G_principal.neighbors(nodo_art):
        if G_principal.nodes[vecino]["tipo"] == "autor":
            autores_art.append(vecino)
    autores_art.sort()

    for pareja in combinations(autores_art, 2):
        if pareja not in compartidos:
            compartidos[pareja] = []
        compartidos[pareja].append(int(indice))

for (autor1, autor2), pubs in compartidos.items():
    G_coautoria.add_edge(
        autor1,
        autor2,
        tipo="coautoria",
        weight=len(pubs),
        articulos=lista_json(pubs)
    )

# Guardar el grado (cantidad de coautores distintos) como atributo
for nodo, grado in G_coautoria.degree():
    G_coautoria.nodes[nodo]["grado"] = grado

## 5. Revisar los resultados

In [ ]:
# Contar nodos por tipo en el grafo principal
conteo_nodos = {}
for _, attrs in G_principal.nodes(data=True):
    t = attrs["tipo"]
    conteo_nodos[t] = conteo_nodos.get(t, 0) + 1

# Contar aristas por tipo en el grafo principal
conteo_aristas = {}
for _, _, attrs in G_principal.edges(data=True):
    t = attrs["tipo"]
    conteo_aristas[t] = conteo_aristas.get(t, 0) + 1

print("--- Grafo principal ---")
print("Artículos:", conteo_nodos.get("articulo", 0))
print("Autores:", conteo_nodos.get("autor", 0))
print("Afiliaciones:", conteo_nodos.get("afiliacion", 0))
print("Palabras clave:", conteo_nodos.get("keyword", 0))
print("Total de nodos:", G_principal.number_of_nodes())
print("Autor-artículo:", conteo_aristas.get("autoria", 0))
print("Autor-afiliación:", conteo_aristas.get("afiliacion", 0))
print("Artículo-palabra clave:", conteo_aristas.get("keyword", 0))
print("Total de aristas:", G_principal.number_of_edges())

aislados = list(nx.isolates(G_coautoria))
componentes = list(nx.connected_components(G_coautoria))
mayor_comp = max(len(c) for c in componentes)
suma_pesos = sum(attrs["weight"] for _, _, attrs in G_coautoria.edges(data=True))

# Verificar que la suma de pesos coincida con las parejas por artículo
por_articulo = datos.groupby("indice").size()
parejas_esperadas = sum(n * (n - 1) // 2 for n in por_articulo)

print("\n--- Subgrafo de coautoría ---")
print("Autores:", G_coautoria.number_of_nodes())
print("Parejas distintas:", G_coautoria.number_of_edges())
print("Autores aislados:", len(aislados))
print("Componentes conexas:", len(componentes))
print("Autores en la componente mayor:", mayor_comp)
print("Suma de pesos:", suma_pesos, "(esperada:", parejas_esperadas, ")")

## 6. Guardar y volver a abrir los grafos

In [ ]:
os.makedirs(carpeta_salida, exist_ok=True)

ruta_principal = carpeta_salida + "/grafo_principal.gml"
ruta_coautoria = carpeta_salida + "/subgrafo_coautoria.gml"

nx.write_gml(G_principal, ruta_principal)
nx.write_gml(G_coautoria, ruta_coautoria)

# Comprobamos que al volver a abrirlos queden iguales
igual_p = nx.utils.graphs_equal(G_principal, nx.read_gml(ruta_principal))
igual_c = nx.utils.graphs_equal(G_coautoria, nx.read_gml(ruta_coautoria))
print("Guardado grafo_principal.gml | Coincide al reabrir:", igual_p)
print("Guardado subgrafo_coautoria.gml | Coincide al reabrir:", igual_c)

# Buscar la pareja con más artículos compartidos
mejor_par = None
max_peso = 0
for a, b, attrs in G_coautoria.edges(data=True):
    if attrs["weight"] > max_peso:
        max_peso = attrs["weight"]
        mejor_par = (a, b, attrs)

if mejor_par is not None:
    a, b, attrs = mejor_par
    print("\nPareja con mayor colaboración:")
    print(G_coautoria.nodes[a]["nombre"], "y", G_coautoria.nodes[b]["nombre"])
    print("Peso:", attrs["weight"])
    print("Índices compartidos:", json.loads(attrs["articulos"]))

## 7. Visualizar los grafos completos

In [ ]:
semilla = 42
np.random.seed(semilla)

# Primero calculamos posiciones sin las palabras clave para que quede más ordenado
nucleo = nx.Graph()
for u, v, attrs in G_principal.edges(data=True):
    if attrs["tipo"] != "keyword":
        nucleo.add_edge(u, v)

pos = nx.spring_layout(nucleo, seed=semilla, iterations=90, weight=None)

# Colocamos cada palabra clave cerca del promedio de sus artículos
for nodo, attrs in G_principal.nodes(data=True):
    if attrs["tipo"] == "keyword":
        vecinos_pos = [pos[v] for v in G_principal.neighbors(nodo)]
        centro = np.mean(vecinos_pos, axis=0)
        pos[nodo] = centro + np.random.normal(0, 0.018, size=2)

estilo = {
    "keyword": ("Palabras clave", "#CF951D", 4),
    "articulo": ("Artículos", "#2563EB", 17),
    "autor": ("Autores", "#138B72", 19),
    "afiliacion": ("Afiliaciones", "#8B5AC2", 62)
}

fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_edges(G_principal, pos, ax=ax, edge_color="#637684", alpha=0.17, width=0.5)

for tipo, (etiqueta, color, tamano) in estilo.items():
    grupo = [n for n, a in G_principal.nodes(data=True) if a["tipo"] == tipo]
    nx.draw_networkx_nodes(
        G_principal, pos, nodelist=grupo, node_color=color,
        node_size=tamano, linewidths=0, label=f"{etiqueta}: {len(grupo):,}", ax=ax
    )

ax.set_title(f"Grafo principal\n{G_principal.number_of_nodes():,} nodos y {G_principal.number_of_edges():,} aristas")
ax.legend(loc="upper left", frameon=False, markerscale=1.5)
ax.axis("off")
fig.tight_layout()
fig.savefig(carpeta_salida + "/grafo_principal.png", dpi=220)
plt.show()

In [ ]:
pos_co = nx.spring_layout(G_coautoria, seed=semilla, iterations=80, weight=None)
tamanos = [15 + G_coautoria.nodes[n]["grado"] * 6 for n in G_coautoria.nodes()]

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(G_coautoria, pos_co, ax=ax, edge_color="#637684", alpha=0.35, width=0.8)
nx.draw_networkx_nodes(G_coautoria, pos_co, node_color="#138B72", node_size=tamanos, linewidths=0, ax=ax)

ax.set_title(f"Subgrafo de coautoría\n{G_coautoria.number_of_nodes():,} autores y {G_coautoria.number_of_edges():,} parejas")
ax.axis("off")
fig.tight_layout()
fig.savefig(carpeta_salida + "/subgrafo_coautoria.png", dpi=220)
plt.show()